# Frequency Guard - Deploy from GitHub to Colab
Deploy your Frequency Guard project from GitHub with ngrok public link

## Step 1: Install Node.js

In [ ]:
%%bash
# Install Node.js 20.x
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt-get install -y nodejs
echo "Node version: $(node --version)"
echo "NPM version: $(npm --version)"

## Step 2: Clone Project from GitHub

In [ ]:
# ⚙️ Configure your GitHub repository
GITHUB_USERNAME = "YOUR_USERNAME"  # Replace with your GitHub username
REPO_NAME = "frequency-guard"      # Replace with your repo name

# Clone repository
!rm -rf /content/{REPO_NAME}
!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git /content/{REPO_NAME}

print("\n✅ Repository cloned successfully!")
print(f"📁 Project location: /content/{REPO_NAME}")
!ls -la /content/{REPO_NAME}

## Step 3: Install Project Dependencies

In [ ]:
%%bash
cd /content/frequency-guard
echo "📦 Installing dependencies..."
npm install
echo "\n✅ Dependencies installed successfully!"

## Step 4: Setup ngrok

In [ ]:
%%bash
# Download and install ngrok
echo "📥 Downloading ngrok..."
wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
tar -xzf ngrok-v3-stable-linux-amd64.tgz
chmod +x ngrok
sudo mv ngrok /usr/local/bin/
echo "✅ ngrok installed: $(ngrok version)"

In [ ]:
# 🔑 Add your ngrok authtoken
# Get your free token from: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

!ngrok authtoken {NGROK_AUTH_TOKEN}
print("✅ ngrok configured successfully!")

## Step 5: Deploy and Get Public Link 🚀

In [ ]:
import subprocess
import time
import requests
from IPython.display import display, HTML, clear_output

print("🚀 Starting Frequency Guard...\n")

# Start Vite dev server
print("⚡ Starting Vite development server...")
vite_process = subprocess.Popen(
    ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "5173"],
    cwd="/content/frequency-guard",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
print("⏳ Waiting for server to initialize...")
time.sleep(12)

# Start ngrok tunnel
print("🌐 Creating ngrok tunnel...")
ngrok_process = subprocess.Popen(
    ["ngrok", "http", "5173", "--log=stdout"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for ngrok to start
time.sleep(5)

# Get ngrok public URL
try:
    response = requests.get("http://localhost:4040/api/tunnels", timeout=5)
    tunnels = response.json()["tunnels"]
    public_url = tunnels[0]["public_url"]
    
    clear_output(wait=True)
    
    print("\n" + "="*70)
    print("🎉 FREQUENCY GUARD IS LIVE!")
    print("="*70)
    print(f"\n🔗 Public URL: {public_url}")
    print("\n📱 Share this link with anyone to access your app")
    print("⏰ Keep this cell running to maintain the connection")
    print("🛑 Click 'Stop' button to shut down the server\n")
    print("="*70 + "\n")
    
    # Display clickable link
    display(HTML(f'''
        <div style="text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 10px; margin: 20px 0;">
            <h1 style="color: white; margin-bottom: 20px;">🎊 Deployment Successful!</h1>
            <a href="{public_url}" target="_blank" 
               style="display: inline-block; padding: 15px 40px; background: white; color: #667eea; 
                      text-decoration: none; border-radius: 5px; font-size: 20px; font-weight: bold;
                      box-shadow: 0 4px 6px rgba(0,0,0,0.2); transition: transform 0.2s;"
               onmouseover="this.style.transform='scale(1.05)'" 
               onmouseout="this.style.transform='scale(1)'">
                🚀 Open Frequency Guard
            </a>
            <p style="color: white; margin-top: 20px; font-size: 14px;">{public_url}</p>
        </div>
    '''))
    
    print("\n✨ Server is running...\n")
    print("📊 Logs:")
    print("-" * 70)
    
    # Keep running and show status
    try:
        vite_process.wait()
    except KeyboardInterrupt:
        print("\n🛑 Shutting down...")
        vite_process.terminate()
        ngrok_process.terminate()
        
except Exception as e:
    print(f"\n❌ Error getting ngrok URL: {e}")
    print("\n🔧 Troubleshooting steps:")
    print("1. Check if ngrok authtoken is valid")
    print("2. Visit ngrok dashboard: https://dashboard.ngrok.com/")
    print("3. Check ngrok inspector: http://localhost:4040")
    print("\n⏳ Waiting a bit longer...")
    time.sleep(5)
    try:
        response = requests.get("http://localhost:4040/api/tunnels", timeout=5)
        tunnels = response.json()["tunnels"]
        public_url = tunnels[0]["public_url"]
        print(f"\n✅ Success! Public URL: {public_url}")
        display(HTML(f'<h2><a href="{public_url}" target="_blank" style="color: #2196F3;">🚀 Open App →</a></h2>'))
        vite_process.wait()
    except:
        print("\n❌ Could not retrieve URL. Please check ngrok dashboard.")

## 📝 Important Notes

### ✅ What to do:
- **Keep Step 5 cell running** to maintain the live connection
- Share the public URL with anyone who needs access
- Access ngrok inspector at: http://localhost:4040 for request logs

### ⚠️ Limitations:
- **Free ngrok tunnels expire after 2 hours** - restart Step 5 to get a new URL
- Colab runtime disconnects after ~12 hours of inactivity
- Don't close this browser tab while the server is running

### 🔧 Troubleshooting:
If something goes wrong:
1. Restart runtime: `Runtime > Restart runtime`
2. Run all cells from Step 1 again
3. Check GitHub repo is public or you have access
4. Verify ngrok token at: https://dashboard.ngrok.com/